In [1]:
import pandas as pd 
import holidays

In [13]:
# Charger le jeu de données
df = pd.read_csv('./dataset/champs_elysees.csv', sep=";")

# Renommer les colonnes pour un accès plus facile
df.rename(columns={
    'Date et heure de comptage': 'datetime',
    'Débit horaire': 'debit_horaire',
    "Taux d'occupation": 'taux_occupation',
    'Libelle noeud amont': 'noeud_amont',
    'Libelle noeud aval': 'noeud_aval'
}, inplace=True)

# Conversion de la colonne 'datetime' en objet datetime de pandas
df['datetime'] = pd.to_datetime(df['datetime'], utc=True)

# --- FEATURE ENGINEERING - DATE ---

# 1. Créer une colonne 'date' qui ne contient que la date (sans l'heure)
# On utilise l'accesseur .dt qui permet d'appliquer des méthodes de date/heure à toute la colonne
df['date'] = df['datetime'].dt.date

# 2. Créer une variable pour le jour de la semaine, adaptée à la prédiction
# La meilleure pratique est le "One-Hot Encoding".
# Cela crée une colonne binaire (0 ou 1) pour chaque jour de la semaine.

# Étape A : Obtenir le nom du jour en français
df['jour_nom'] = df['datetime'].dt.day_name(locale='fr_FR')

# Étape B : Créer les colonnes binaires (dummies)
dummies_jours = pd.get_dummies(df['jour_nom'], prefix='jour', dtype=int)

# Étape C : Joindre ces nouvelles colonnes au DataFrame principal
df = pd.concat([df, dummies_jours], axis=1)

# -------------------------------

# Définir la colonne 'datetime' comme index du DataFrame
df.set_index('datetime', inplace=True)

# Trier les données par date, crucial pour une série temporelle correcte
df.sort_index(inplace=True)

# Afficher les premières lignes pour vérifier la création des nouvelles colonnes
# (j'affiche quelques-unes des nouvelles colonnes pour l'exemple)
print("Aperçu du DataFrame avec les nouvelles colonnes de date et de jour :")
print(df[['date', 'jour_nom', 'jour_Lundi', 'jour_Mardi', 'jour_Samedi']].head())

Aperçu du DataFrame avec les nouvelles colonnes de date et de jour :
                                 date  jour_nom  jour_Lundi  jour_Mardi  \
datetime                                                                  
2024-09-01 03:00:00+00:00  2024-09-01  Dimanche           0           0   
2024-09-01 04:00:00+00:00  2024-09-01  Dimanche           0           0   
2024-09-01 05:00:00+00:00  2024-09-01  Dimanche           0           0   
2024-09-01 06:00:00+00:00  2024-09-01  Dimanche           0           0   
2024-09-01 07:00:00+00:00  2024-09-01  Dimanche           0           0   

                           jour_Samedi  
datetime                                
2024-09-01 03:00:00+00:00            0  
2024-09-01 04:00:00+00:00            0  
2024-09-01 05:00:00+00:00            0  
2024-09-01 06:00:00+00:00            0  
2024-09-01 07:00:00+00:00            0  


In [ ]:
from datetime import date

# --- 1. DÉFINITION DES LISTES DE DATES ---

# Liste exhaustive des jours fériés (format YYYY, M, D)
jours_feries_liste = [
    date(2024, 11, 1),   # Toussaint
    date(2024, 11, 11),  # Armistice 1918
    date(2024, 12, 25),  # Noël
    date(2025, 1, 1),    # Nouvel An
    date(2025, 4, 21),   # Lundi de Pâques
    date(2025, 5, 1),    # Fête du Travail
    date(2025, 5, 8),    # Victoire 1945
    date(2025, 5, 29),   # Ascension
    date(2025, 6, 9),    # Lundi de Pentecôte
    date(2025, 7, 14),   # Fête Nationale
    date(2025, 8, 15),   # Assomption
]

# Liste des périodes de vacances scolaires pour la Zone C (Paris)
vacances_scolaires_periodes = [
    ('2024-10-19', '2024-11-03'), # Toussaint 2024
    ('2024-12-21', '2025-01-05'), # Noël 2024
    ('2025-02-15', '2025-03-02'), # Hiver 2025
    ('2025-04-12', '2025-04-27'), # Printemps 2025
    ('2025-07-05', '2025-08-31'), # Été 2025 (jusqu'à la rentrée)
    ('2025-10-18', '2025-10-31')  # Toussaint 2025 (jusqu'à la fin de la période demandée)
]


# --- 2. CRÉATION DES VARIABLES BINAIRES ---

# S'assurer que l'index est bien de type datetime pour la comparaison
if not isinstance(df.index, pd.DatetimeIndex):
    df['datetime'] = pd.to_datetime(df['datetime'], utc=True)
    df.set_index('datetime', inplace=True)
    df.sort_index(inplace=True)

# Créer une colonne 'date' simplifiée pour les comparaisons
df['date'] = df.index.date

# Variable binaire 'jour_ferie'
df['jour_ferie'] = df['date'].isin(jours_feries_liste).astype(int)

# Variable binaire 'veille_jour_ferie'
# On crée une liste des veilles à partir de la liste des jours fériés
veilles_feries_liste = [d - pd.Timedelta(days=1) for d in jours_feries_liste]
df['veille_jour_ferie'] = df['date'].isin(veilles_feries_liste).astype(int)

# Variable binaire 'vacances_scolaires'
# On initialise la colonne à 0
df['vacances_scolaires'] = 0
# On parcourt chaque période de vacances et on met à 1 les jours correspondants
for debut, fin in vacances_scolaires_periodes:
    mask = (df['date'] >= pd.to_datetime(debut).date()) & (df['date'] <= pd.to_datetime(fin).date())
    df.loc[mask, 'vacances_scolaires'] = 1

# --- 3. VÉRIFICATION ---
print("Aperçu des nouvelles colonnes créées :")
# On affiche un extrait autour d'un jour férié pour vérifier
print(df.loc['2024-10-31':'2024-11-02'][['date', 'jour_ferie', 'veille_jour_ferie', 'vacances_scolaires']])

print("\nAperçu pendant les vacances de Noël :")
print(df.loc['2024-12-24':'2024-12-26'][['date', 'jour_ferie', 'veille_jour_ferie', 'vacances_scolaires']])


Aperçu des nouvelles colonnes créées :
                                 date  jour_ferie  veille_jour_ferie  \
datetime                                                               
2024-10-31 00:00:00+00:00  2024-10-31           0                  1   
2024-10-31 01:00:00+00:00  2024-10-31           0                  1   
2024-10-31 02:00:00+00:00  2024-10-31           0                  1   
2024-10-31 03:00:00+00:00  2024-10-31           0                  1   
2024-10-31 04:00:00+00:00  2024-10-31           0                  1   
...                               ...         ...                ...   
2024-11-02 19:00:00+00:00  2024-11-02           0                  0   
2024-11-02 20:00:00+00:00  2024-11-02           0                  0   
2024-11-02 21:00:00+00:00  2024-11-02           0                  0   
2024-11-02 22:00:00+00:00  2024-11-02           0                  0   
2024-11-02 23:00:00+00:00  2024-11-02           0                  0   

                        

In [25]:
import pandas as pd
from astral.location import LocationInfo
from astral.sun import sun

# --- 1. CHARGEMENT ET PRÉPARATION INITIALE ---

# Colonnes à garder, y compris le nom de la station pour le filtrage
colonnes_utiles = [
    'NOM_USUEL',   # Garder le nom de la station
    'AAAAMMJJHH',
    'T',
    'RR1',
    'FF',
    'NEIGETOT',
    'INS'
]

# Charger le jeu de données
df_meteo = pd.read_csv('./dataset/H_75_latest-2024-2025.csv', sep=";", usecols=colonnes_utiles)

# Renommer les colonnes
df_meteo.rename(columns={
    'AAAAMMJJHH': 'datetime',
    'T': 'temperature',
    'RR1': 'precipitation_mm',
    'FF': 'vent_vitesse_ms',
    'NEIGETOT': 'neige_cm',
    'INS': 'ensoleillement_min'
}, inplace=True)

# Conversion de la colonne 'datetime' au format datetime
df_meteo['datetime'] = pd.to_datetime(df_meteo['datetime'], format='%Y%m%d%H', utc=True)

# Ajustement des unités
df_meteo['temperature'] = df_meteo['temperature'] / 10.0
df_meteo['precipitation_mm'] = df_meteo['precipitation_mm'] / 10.0
df_meteo['vent_vitesse_ms'] = df_meteo['vent_vitesse_ms'] / 10.0

# --- NOUVELLE ÉTAPE : SÉLECTION DE LA MEILLEURE STATION ---

# 1. Calculer le nombre de valeurs manquantes pour chaque station
# On somme les NaN sur toutes les colonnes pour chaque station
missing_values_per_station = df_meteo.isnull().groupby(df_meteo['NOM_USUEL']).sum().sum(axis=1)
print("--- Nombre de valeurs manquantes par station ---")
print(missing_values_per_station)

# 2. Identifier le nom de la station avec le minimum de valeurs manquantes
best_station = missing_values_per_station.idxmin()
print(f"\nLa meilleure station (le moins de NaN) est : '{best_station}'")

# 3. Filtrer le DataFrame pour ne garder que les lignes de cette station
df_meteo = df_meteo[df_meteo['NOM_USUEL'] == best_station].copy()

# 4. Supprimer la colonne 'NOM_USUEL' qui est maintenant inutile
df_meteo.drop(columns=['NOM_USUEL'], inplace=True)

print(f"\nLe DataFrame a été filtré pour ne conserver que les données de '{best_station}'.")

# --- 2. FILTRAGE ET MISE EN FORME (sur le df nettoyé) ---

# Définir datetime comme index. Il ne devrait plus y avoir de doublons.
df_meteo.set_index('datetime', inplace=True)

# Trier l'index par ordre chronologique
df_meteo.sort_index(inplace=True)

# Filtrer pour ne garder que la période d'intérêt
df_meteo = df_meteo.loc['2024-09-01':'2025-10-31']


# --- 3. CRÉATION DES VARIABLES BINAIRES ---

# Variable binaire 'il_pleut'
df_meteo['il_pleut'] = (df_meteo['precipitation_mm'] > 0).astype(int)

# Variable binaire 'il_fait_jour'
loc = LocationInfo("Paris", "France", "Europe/Paris", 48.8566, 2.3522)
unique_dates = df_meteo.index.normalize().unique()
sun_times = {
    day: sun(loc.observer, date=day, tzinfo=loc.timezone)
    for day in unique_dates
}

def is_daylight(timestamp):
    day = timestamp.normalize()
    if day in sun_times:
        return sun_times[day]['sunrise'] <= timestamp <= sun_times[day]['sunset']
    return False

df_meteo['il_fait_jour'] = df_meteo.index.to_series().apply(is_daylight).astype(int)


# --- 4. VÉRIFICATION ---
print("\nAperçu du DataFrame météo final (une seule station) :")
# La sortie ne devrait plus avoir de datetime en double
print(df_meteo.head())

print("\nDescription statistique des nouvelles données :")
print(df_meteo[['temperature', 'precipitation_mm', 'vent_vitesse_ms', 'neige_cm', 'ensoleillement_min']].describe())

--- Nombre de valeurs manquantes par station ---
NOM_USUEL
LARIBOISIERE               48377
LONGCHAMP                  16302
LUXEMBOURG                 48559
PARIS-MONTSOURIS            2461
PARIS-MONTSOURIS-DOUBLE    64432
TOUR EIFFEL                48768
dtype: int64

La meilleure station (le moins de NaN) est : 'PARIS-MONTSOURIS'

Le DataFrame a été filtré pour ne conserver que les données de 'PARIS-MONTSOURIS'.

Aperçu du DataFrame météo final (une seule station) :
                           precipitation_mm  vent_vitesse_ms  temperature  \
datetime                                                                    
2024-09-01 00:00:00+00:00               0.0             0.13         1.89   
2024-09-01 01:00:00+00:00               0.0             0.15         1.88   
2024-09-01 02:00:00+00:00               0.0             0.19         1.88   
2024-09-01 03:00:00+00:00               0.0             0.17         1.95   
2024-09-01 04:00:00+00:00               0.0             0.26    

In [30]:
# --- 1. FUSION DES DATAFRAMES ---

print(f"Taille du DataFrame de trafic avant fusion : {df.shape}")
print(f"Taille du DataFrame météo avant fusion :    {df_meteo.shape}")

# On utilise df.join() pour fusionner les données météo dans le DataFrame de trafic.
# La jointure se fait sur l'index 'datetime' qui est commun aux deux.
# 'how=left' garantit que toutes les lignes originales du trafic sont conservées.
df_final = df.join(df_meteo, how='left')

print(f"Taille du DataFrame final après fusion :     {df_final.shape}")


# --- 2. VÉRIFICATION DE LA FUSION ---

# Afficher les premières lignes du DataFrame fusionné pour voir les nouvelles colonnes
print("\nAperçu du DataFrame final fusionné :")
print(df_final[['debit_horaire', 'taux_occupation', 'temperature', 'precipitation_mm', 'il_pleut']].head())

# Vérifier s'il y a des valeurs manquantes dans les colonnes météo après la fusion.
# Un petit nombre de NaN est possible et normal, un grand nombre pourrait indiquer un problème.
print("\nNombre de valeurs manquantes pour les colonnes météo dans le DataFrame final :")
print(df_final[['temperature', 'precipitation_mm', 'vent_vitesse_ms', 'neige_cm', 'ensoleillement_min', 'il_pleut', 'il_fait_jour']].isnull().sum())

Taille du DataFrame de trafic avant fusion : (9266, 26)
Taille du DataFrame météo avant fusion :    (10224, 7)
Taille du DataFrame final après fusion :     (9266, 33)

Aperçu du DataFrame final fusionné :
                           debit_horaire  taux_occupation  temperature  \
datetime                                                                 
2024-09-01 03:00:00+00:00          532.0          8.08889         1.95   
2024-09-01 04:00:00+00:00          331.0          3.91500         1.99   
2024-09-01 05:00:00+00:00          272.0          2.69167         1.97   
2024-09-01 06:00:00+00:00          191.0          2.23612         1.98   
2024-09-01 07:00:00+00:00          201.0          2.63333         2.13   

                           precipitation_mm  il_pleut  
datetime                                               
2024-09-01 03:00:00+00:00              0.00         0  
2024-09-01 04:00:00+00:00              0.00         0  
2024-09-01 05:00:00+00:00              0.02         

In [36]:
# --- 1. ENREGISTREMENT DU DATAFRAME FINAL ---

# Définir le nom du fichier de sortie
nom_fichier_sortie = 'dataset/enriched_dataset.csv'

# Enregistrer le DataFrame en fichier CSV
# sep=';' : Utilise le point-virgule comme séparateur, ce qui est courant en France et bien géré par Excel.
# index=True : Conserve la colonne 'datetime' comme la première colonne du fichier, ce qui est essentiel.
# encoding='utf-8-sig' : Assure une bonne compatibilité avec les caractères spéciaux (accents) et une ouverture facile dans Excel.
df_final.to_csv(nom_fichier_sortie, sep=';', index=True, encoding='utf-8-sig')

In [35]:
# Colonnes sur lesquelles nous allons créer les features de lag
colonnes_cibles = ['debit_horaire', 'taux_occupation']

# --- 1. MOYENNE GLISSANTE DES 3 DERNIÈRES HEURES ---

for col in colonnes_cibles:
    # On utilise .rolling(window=3) pour définir une fenêtre de 3 périodes (heures).
    # .shift(1) est CRUCIAL : il décale la fenêtre d'une heure dans le passé pour
    # s'assurer qu'on n'utilise que des données passées pour prédire le futur (évite la fuite de données).
    # min_periods=1 permet de calculer la moyenne même s'il n'y a qu'une ou deux valeurs disponibles au début.
    nom_nouvelle_colonne = f'{col}_moyenne_3h'
    df_final[nom_nouvelle_colonne] = df_final[col].shift(1).rolling(window=3, min_periods=1).mean()


# --- 2. MOYENNE DES 3 JOURS PRÉCÉDENTS (MÊME HEURE) ---

for col in colonnes_cibles:
    # Créer les lags individuels pour J-1, J-2, J-3 (24h, 48h, 72h)
    lag_j1 = df_final[col].shift(24)
    lag_j2 = df_final[col].shift(48)
    lag_j3 = df_final[col].shift(72)
    
    # Calculer la moyenne de ces trois lags
    nom_nouvelle_colonne = f'{col}_moyenne_3j_meme_heure'
    df_final[nom_nouvelle_colonne] = (lag_j1 + lag_j2 + lag_j3) / 3


# --- 3. MOYENNE DES 3 SEMAINES PRÉCÉDENTES (MÊME JOUR, MÊME HEURE) ---

for col in colonnes_cibles:
    # Le lag pour une semaine est de 24 * 7 = 168 heures
    lag_s1 = df_final[col].shift(168)
    lag_s2 = df_final[col].shift(336) # 168 * 2
    lag_s3 = df_final[col].shift(504) # 168 * 3
    
    # Calculer la moyenne de ces trois lags hebdomadaires
    nom_nouvelle_colonne = f'{col}_moyenne_3s_meme_jour_heure'
    df_final[nom_nouvelle_colonne] = (lag_s1 + lag_s2 + lag_s3) / 3

    
# --- 4. VÉRIFICATION ---

# Lister toutes les nouvelles colonnes créées
nouvelles_colonnes = [f'{col}_moyenne_3h' for col in colonnes_cibles] + \
                   [f'{col}_moyenne_3j_meme_heure' for col in colonnes_cibles] + \
                   [f'{col}_moyenne_3s_meme_jour_heure' for col in colonnes_cibles]

print("Aperçu des nouvelles variables de lag créées :")
# .iloc[504:] est utilisé pour afficher une partie du df où tous les lags sont calculés et non nuls
print(df_final[colonnes_cibles + nouvelles_colonnes].iloc[504:].head())

Aperçu des nouvelles variables de lag créées :
                           debit_horaire  taux_occupation  \
datetime                                                    
2024-09-22 09:00:00+00:00          600.0          9.50611   
2024-09-22 10:00:00+00:00          703.0         12.56222   
2024-09-22 11:00:00+00:00          782.0         13.95167   
2024-09-22 12:00:00+00:00          851.0         15.21556   
2024-09-22 13:00:00+00:00          708.0         15.37556   

                           debit_horaire_moyenne_3h  \
datetime                                              
2024-09-22 09:00:00+00:00                366.333333   
2024-09-22 10:00:00+00:00                457.333333   
2024-09-22 11:00:00+00:00                584.000000   
2024-09-22 12:00:00+00:00                695.000000   
2024-09-22 13:00:00+00:00                778.666667   

                           taux_occupation_moyenne_3h  \
datetime                                                
2024-09-22 09:00:00+00:00